# components

> Oat's components as FastHTML functions, with the same names as `fasthtml.pico` where Pico has one.

In [ ]:
#| default_exp components

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import hashlib, json
from fastcore.meta import delegates
from fastcore.xml import FT, NotStr
from fasthtml.common import *
from fasthtml.components import Ot_dropdown, Ot_tabs, Ot_upload, Ot_taginput
from fhoat.core import THEMES

Plain tags need no helper: `Button`, `A`, `Input`, `Select`, `Textarea`, `Table`, `Dialog`, `Details`, `Progress`, `Meter` and the
headings are styled as they are. A `Div` has no look of its own; give it a class (`card`, `row`, `hstack`, `vstack`, `container`) or use
the functions below. An accordion is plain `Details(Summary('Title'), P('Body'))`.

In [ ]:
#| export
def _cls(*c): return ' '.join(o for o in c if o) or None
def _uid(p, *parts): return f"{p}-{hashlib.md5(''.join(map(to_xml, parts)).encode(), usedforsecurity=False).hexdigest()[:8]}"
def _js(d): return json.dumps(d).replace('<', '\\u003c').replace('>', '\\u003e').replace('&', '\\u0026')
_SIZE = dict(sm='small', small='small', lg='large', large='large')

## Pico's names

In [ ]:
#| export
@delegates(ft_hx, keep=True)
def Card(*c, header=None, footer=None, cls='', **kw) -> FT:
    "An Oat card: an Article with an optional Header and Footer, each ruled off."
    if header: c = (Header(header),) + c
    if footer: c += (Footer(footer),)
    return Article(*c, cls=_cls('card ot-card', cls), **kw)

@delegates(ft_hx, keep=True)
def Group(*c, cls='', **kw) -> FT:
    "Joined inputs and buttons: a Fieldset with class `group`."
    return Fieldset(*c, cls=_cls('group', cls), role='group', **kw)

@delegates(ft_hx, keep=True)
def Search(*c, **kw) -> FT:
    "A Form with role `search` (pico's name; it hides FastHTML's `<search>` tag)."
    return Form(*c, role='search', **kw)

@delegates(ft_hx, keep=True)
def Grid(*c, col_min=None, cls='', **kw) -> FT:
    "Equal columns that wrap below `col_min` (a CSS length, default 14rem); each child in a Div, as pico's Grid."
    c = tuple(o if isinstance(o, list) else Div(o) for o in c)
    return Div(*c, cls=_cls('ot-grid', cls), style=f'--min:{col_min}' if col_min else None, **kw)

def Row(*c, cls='', **kw) -> FT:
    "Oat's 12-column row (4 columns on a phone)."
    return Div(*c, cls=_cls('row', cls), **kw)

def Column(*c, span=None, offset=None, cls='', **kw) -> FT:
    "A column in a Row spanning `span` of 12 (FastHTML's `Col` is the `<col>` tag, so this is `Column`)."
    return Div(*c, cls=_cls(f'col-{span}' if span else 'col', offset and f'offset-{offset}', cls), **kw)

@delegates(ft_hx, keep=True)
def Container(*c, cls='', **kw) -> FT:
    "A Main with class `container`."
    return Main(*c, cls=_cls('container', cls), **kw)

@delegates(ft_hx, keep=True)
def DialogX(*c, open=None, header=None, footer=None, id=None, closedby='any', form=False, **kw) -> FT:
    "An Oat dialog: header, body and footer; `closedby='any'` closes on a backdrop click; `form` wraps the parts in a dialog form."
    parts = (Header(header) if header else None, Div(*c), Footer(footer) if footer else None)
    return Dialog(Form(*parts, method='dialog') if form else parts, open=open, id=id, closedby=closedby, **kw)

def OatBusy():
    "Mark the element of a running htmx request `aria-busy`, which Oat draws as a spinner."
    elt = '(event.detail.ctx?.sourceElement || event.detail.elt)'
    return (HtmxOn('beforeRequest', f"{elt}.setAttribute('aria-busy', 'true' )"),
            HtmxOn('afterRequest', f"{elt}.setAttribute('aria-busy', 'false')"))

In [ ]:
assert 'class="card ot-card"' in to_xml(Card('b', header='h', footer='f')) and '<header>h</header>' in to_xml(Card('b', header='h'))
assert 'class="group"' in to_xml(Group(Input())) and 'role="search"' in to_xml(Search())
assert '--min:10rem' in to_xml(Grid('a', 'b', col_min='10rem')) and to_xml(Grid('a')).count('<div>') == 1
assert 'class="col-8 offset-2"' in to_xml(Column(span=8, offset=2)) and 'class="row"' in to_xml(Row())
assert 'closedby="any"' in to_xml(DialogX('b', id='d')) and 'closedby="none"' in to_xml(DialogX('b', closedby='none'))
assert 'method="dialog"' in to_xml(DialogX('b', form=True))

## Dialogs and buttons

In [ ]:
#| export
def DialogOpen(*c, id, cls='', **kw) -> FT:
    "A button that opens dialog `id` as a modal (invoker commands; Oat polyfills them)."
    return Button(*c, type='button', commandfor=id, command='show-modal', cls=_cls(cls), **kw)

def DialogClose(*c, id=None, cls='outline', **kw) -> FT:
    "A button that closes dialog `id`, or the dialog it sits in when there is no `id`."
    if id: return Button(*(c or ('Close',)), type='button', commandfor=id, command='close', cls=_cls(cls), **kw)
    return Button(*(c or ('Close',)), type='button', onclick="this.closest('dialog').close()", cls=_cls(cls), **kw)

def _btn(variant, look, size, icon): return dict(data_variant=variant) if variant else {}, (look, _SIZE.get(size, size), icon and 'icon')

def Btn(*c, variant=None, look=None, size=None, icon=False, cls='', **kw) -> FT:
    "A button: `variant` secondary|danger, `look` outline|ghost, `size` small|large (or sm|lg), `icon` for a square one."
    a, k = _btn(variant, look, size, icon)
    return Button(*c, cls=_cls(*k, cls), **a, **kw)

def ABtn(*c, href='#', variant=None, look=None, size=None, icon=False, cls='', **kw) -> FT:
    "A link drawn as a button."
    a, k = _btn(variant, look, size, icon)
    return A(*c, href=href, cls=_cls('button', *k, cls), **a, **kw)

def ButtonGroup(*c, **kw) -> FT:
    "Connected buttons: Oat's `menu.buttons`, each button in an Li."
    return Menu(*[Li(o) for o in c], cls='buttons', **kw)

def tip(text, placement=None) -> dict:
    "Attrs for an Oat tooltip, to splat into any tag: `Btn('Save', **tip('Saves the draft', 'bottom'))`."
    return dict(title=text) | (dict(data_tooltip_placement=placement) if placement else {})

In [ ]:
assert 'commandfor="d"' in to_xml(DialogOpen('o', id='d')) and 'command="close"' in to_xml(DialogClose(id='d'))
assert 'data-variant="danger"' in to_xml(Btn('x', variant='danger')) and 'class="outline small"' in to_xml(Btn('x', look='outline', size='sm'))
assert 'class="button ghost"' in to_xml(ABtn('a', look='ghost')) and 'class="buttons"' in to_xml(ButtonGroup(Btn('a'), Btn('b')))
assert 'data-tooltip-placement="bottom"' in to_xml(Btn('s', **tip('Save', 'bottom')))

## Content

In [ ]:
#| export
def Badge(*c, variant=None, look=None, cls='', **kw) -> FT:
    "A pill: `variant` secondary|success|warning|danger, `look='outline'`."
    return Span(*c, cls=_cls('badge', look, cls), data_variant=variant, **kw)

def Alert(*c, title=None, variant=None, **kw) -> FT:
    "A callout: `variant` success|warning|danger|error."
    return Div(Strong(title) if title else None, *c, role='alert', data_variant=variant, **kw)

def Avatar(src=None, initials=None, name='', size=None, **kw) -> FT:
    "A round avatar: an image or initials; `name` labels it; `size` small|large."
    inner = Img(src=src, alt='') if src else Abbr(initials, title=name or None)
    return Figure(inner, data_variant='avatar', cls=_SIZE.get(size, size), aria_label=name or None, **kw)

def AvatarGroup(*avatars, size=None, label='', **kw) -> FT:
    "Overlapping avatars; `size` applies to all."
    return Figure(*avatars, data_variant='avatar', role='group', cls=_SIZE.get(size, size), aria_label=label or None, **kw)

def Skeleton(n=3, box=False, **kw) -> FT:
    "Loading placeholders: `n` lines, after a box when `box`."
    return Div(Div(role='status', cls='skeleton box') if box else None, *[Div(role='status', cls='skeleton line') for _ in range(n)], **kw)

def Spinner(*c, size=None, overlay=False, **kw) -> FT:
    "A busy spinner; with `overlay` it covers and dims its children."
    sp = ' '.join(o for o in (_SIZE.get(size, size), overlay and 'overlay') if o) or None
    return Div(*c, aria_busy='true', data_spinner=sp, **kw)

In [ ]:
a = to_xml(Alert('x', title='T', variant='warning'))
assert 'role="alert" data-variant="warning"' in a and '<strong>T</strong>x</div>' in a
assert '<abbr title="Jane Doe">JD</abbr>' in to_xml(Avatar(initials='JD', name='Jane Doe')) and 'role="group"' in to_xml(AvatarGroup(Avatar(initials='A')))
assert 'data-spinner="small overlay"' in to_xml(Spinner(P('x'), size='sm', overlay=True)) and 'class="badge outline"' in to_xml(Badge('b', look='outline'))

## Forms

In [ ]:
#| export
def Field(label, inp, hint=None, error=None, invalid=False, **kw) -> FT:
    "A labelled input with a hint (linked by `aria-describedby`) and an error shown when `invalid`."
    base = inp.attrs.get('id') or inp.attrs.get('name') or _uid('f', label)
    ids = [i for i, on in ((f'{base}-hint', hint), (f'{base}-err', error)) if on]
    if ids: inp.attrs['aria-describedby'] = ' '.join(ids)
    if invalid: inp.attrs['aria-invalid'] = 'true'
    return Div(Label(label, inp), Small(hint, data_hint=True, id=f'{base}-hint') if hint else None,
               Small(error, cls='error', id=f'{base}-err') if error else None, data_field=True, **kw)

def Switch(label, **kw) -> FT:
    "A toggle switch."
    return Label(Input(type='checkbox', role='switch', **kw), label)

def Upload(name='files', label='Files', hint='Drop files here or click to choose', multiple=True, **kw) -> FT:
    "Oat's drop zone for files; the chosen names show as badges."
    return Ot_upload(Div(Input(type='file', name=name, multiple=multiple, hidden=True), Strong(label),
                         Div(Small(hint, data_hint=True), data_files=True), data_field=True, cls='vstack'), **kw)

def TagInput(name='tags', value='', placeholder='Add tags…', **kw) -> FT:
    "Oat's tag entry: Enter or comma adds a tag; a hidden input carries the tags to the form."
    sync = "this.nextElementSibling.value=this.value.join(',')"
    return Ot_taginput(Input(placeholder=placeholder), value=value or None, oninput=sync, **kw), Input(type='hidden', name=name, value=value)

In [ ]:
f = to_xml(Field('Email', Input(name='email'), hint='h', error='bad', invalid=True))
assert 'aria-describedby="email-hint email-err"' in f and 'aria-invalid="true"' in f and 'id="email-err"' in f
assert 'role="switch"' in to_xml(Switch('s')) and '<ot-upload>' in to_xml(Upload()) and 'name="tags"' in to_xml(TagInput())

## Tabs, menus and the sidebar

In [ ]:
#| export
def Tabs(*tabs, anchor=None, **kw) -> FT:
    "Tabs from (label, content) pairs; `anchor` deep-links the active tab in the URL hash."
    ids = [f'{anchor}-{i}' if anchor else None for i in range(len(tabs))]
    return Ot_tabs(Div(*[Button(l, role='tab', type='button', id=i) for (l, _), i in zip(tabs, ids)], role='tablist'),
                   *[Div(c, role='tabpanel') for _, c in tabs], data_anchor=anchor, **kw)

def _item(o):
    if isinstance(o, FT) and o.tag == 'hr': return o
    if isinstance(o, FT): return o(role='menuitem', cls=o.attrs.get('class') or 'ghost')
    return Button(o, role='menuitem', type='button', cls='ghost')

def Dropdown(trigger, *items, id=None, cls='', **kw) -> FT:
    "A menu under `trigger` (a string or a Button); items are links, buttons, strings or `Hr()`."
    id = id or _uid('menu', trigger, *items)
    t = trigger(popovertarget=id) if isinstance(trigger, FT) else Button(trigger, popovertarget=id, cls='outline', type='button')
    return Ot_dropdown(t, Menu(*map(_item, items), popover=True, id=id), cls=_cls(cls), **kw)

def Sidebar(nav, *main, header=None, footer=None, topnav=None, always=False, **kw) -> FT:
    "Oat's sidebar layout: `nav` in an aside (with `footer`), `main` beside it, an optional sticky `topnav`."
    return Div(Nav(topnav, data_topnav=True) if topnav else None, Aside(header and Header(header), Nav(nav), footer and Footer(footer), data_sidebar=True),
               Main(*main), data_sidebar_layout='always' if always else True, **kw)

def SidebarToggle(*c, cls='ghost icon small', **kw) -> FT:
    "The button that opens and closes the sidebar on small screens."
    return Button(*(c or ('☰',)), type='button', data_sidebar_toggle=True, cls=cls, aria_label='Menu', **kw)

In [ ]:
t = to_xml(Tabs(('A', 'a'), ('B', 'b'), anchor='t'))
assert t.count('role="tab"') == 2 and 'id="t-1"' in t and 'data-anchor="t"' in t
m = to_xml(Dropdown('Menu', A('a', href='#'), Hr(), 'b'))
assert m == to_xml(Dropdown('Menu', A('a', href='#'), Hr(), 'b')) and m.count('role="menuitem"') == 2 and 'class="ghost"' in m
s = to_xml(Sidebar(Ul(Li(A('Home', href='/'))), P('main'), topnav=SidebarToggle()))
assert 'data-sidebar-layout' in s and 'data-sidebar' in s and 'data-topnav' in s and 'data-sidebar-toggle' in s

## Page chrome and the reader's controls

The controls carry `data-set="key=value"` or `data-cycle="key:a,b,c"`; fhoat.js handles the clicks, so there is no inline script and a
strict CSP still works. Each control takes your own `icon`.

In [ ]:
#| export
def _svg(body, cls=''):
    "A Lucide icon (ISC) from its inner markup."
    return NotStr(f'<svg class="{cls}" viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2" '
                  f'stroke-linecap="round" stroke-linejoin="round" aria-hidden="true">{body}</svg>')
_SUN = '<circle cx="12" cy="12" r="4"/><path d="M12 2v2M12 20v2m-7.07-14.07 1.41 1.41m11.32 11.32 1.41 1.41M2 12h2m16 0h2M6.34 17.66l-1.41 1.41M19.07 4.93l-1.41 1.41"/>'
_MOON = '<path d="M12 3a6 6 0 0 0 9 9 9 9 0 1 1-9-9Z"/>'
_AUTO = '<rect width="20" height="14" x="2" y="3" rx="2"/><path d="M8 21h8M12 17v4"/>'
_CASE = '<path d="m15 16 2.54-7.63a.5.5 0 0 1 .92 0L21 16M15.5 14h5M3 16l4-10 4 10M4.2 13h5.6"/>'
_PALETTE = ('<circle cx="13.5" cy="6.5" r=".5" fill="currentColor"/><circle cx="17.5" cy="10.5" r=".5" fill="currentColor"/>'
            '<circle cx="8.5" cy="7.5" r=".5" fill="currentColor"/><circle cx="6.5" cy="12.5" r=".5" fill="currentColor"/>'
            '<path d="M12 2C6.5 2 2 6.5 2 12s4.5 10 10 10c.93 0 1.65-.75 1.65-1.69 0-.44-.18-.84-.44-1.13-.29-.29-.44-.65-.44-1.12a1.64 1.64 0 0 1 '
            '1.67-1.67h2c3.05 0 5.55-2.5 5.55-5.55C21.97 6.01 17.46 2 12 2z"/>')
RADIUS_OPTS = dict(none='Sharp', sm='Subtle', md='Round', lg='Rounder')
SHADOW_OPTS = dict(none='Flat', sm='Soft', md='Medium', lg='Deep')

In [ ]:
#| export
def ModeToggle(auto=None, light=None, dark=None, order=('auto', 'light', 'dark'), **kw) -> FT:
    "One button cycling the modes in `order`; it shows the current mode's icon."
    ics = [Span(o, cls=k) if o is not None else _svg(d, k) for k, o, d in (('auto', auto, _AUTO), ('light', light, _SUN), ('dark', dark, _MOON))]
    return Button(*ics, type='button', data_cycle='mode:' + ','.join(order), cls='ghost icon small ot-mode', aria_label='Colour mode', **kw)

def FontToggle(sizes=('', 'lg', 'xl'), icon=None, **kw) -> FT:
    "One button cycling the text size through `sizes` ('' is the app default; sm|lg|xl)."
    return Button(icon if icon is not None else _svg(_CASE), type='button', data_cycle='size:' + ','.join(sizes), cls='ghost icon small ot-size',
                  aria_label='Text size', **kw)

def _opts(k, opts): return Div(*[Button(lbl, type='button', role='menuitem', cls='outline small', data_set=f'{k}={v}') for v, lbl in opts.items()], cls='ot-opts')

def ThemePicker(themes=None, radii=RADIUS_OPTS, shadows=SHADOW_OPTS, title='Theme', id='ot-themes', icon=None, **kw) -> FT:
    "A menu that sets the reader's theme, corners and shadows; `radii=None` or `shadows=None` drops a row; give each picker its own `id`."
    sw = [Button(type='button', role='menuitem', cls='ot-swatch', style=f'--sw:{c}', aria_label=n, title=n, data_set=f'theme={n}')
          for n, c in (themes or THEMES).items()]
    menu = Div(H6(title), Div(*sw, cls='ot-swatches'), *((H6('Corners'), _opts('radius', radii)) if radii else ()),
               *((H6('Shadows'), _opts('shadow', shadows)) if shadows else ()), popover=True, id=id, cls='ot-picker')
    btn = Button(icon if icon is not None else _svg(_PALETTE), popovertarget=id, type='button', cls='ghost icon small', aria_label=title)
    return Ot_dropdown(btn, menu, **kw)

In [ ]:
#| export
def _link(o): return A(o[0], href=o[1]) if isinstance(o, tuple) else o

def Navbar(brand, *links, end=None, href='/', cls='', **kw) -> FT:
    "A sticky glass bar: brand, links as (text, href) or FT, then `end` items (default: the mode toggle)."
    end = (ModeToggle(),) if end is None else (end if isinstance(end, (tuple, list)) else (end,))
    return Header(Nav(A(brand, href=href, cls='brand'), Ul(*[Li(_link(l)) for l in links]) if links else None,
                      Div(*end, cls='end')), cls=_cls('ot-nav', cls), **kw)

def Hero(title, *c, sub=None, eyebrow=None, actions=(), cls='', **kw) -> FT:
    "A centred page opener: eyebrow, title, subtitle, then actions."
    return Section(Span(eyebrow, cls='eyebrow') if eyebrow else None, H1(title), P(sub) if sub else None, *c,
                   Div(*actions, cls='hstack') if actions else None, cls=_cls('hero', cls), **kw)

def PageFooter(*c, cls='', **kw) -> FT:
    "A quiet page footer."
    return Footer(Div(*c), cls=_cls('ot-footer', cls), **kw)

In [ ]:
p = to_xml(ThemePicker())
assert p.count('data-set="theme=') == len(THEMES) and 'data-set="radius=lg"' in p and 'onclick' not in p
assert 'data-cycle="mode:auto,dark,light"' in to_xml(ModeToggle(order=('auto', 'dark', 'light'))) and 'data-cycle="size:,lg,sm"' in to_xml(FontToggle(('', 'lg', 'sm')))
assert 'class="ot-nav"' in to_xml(Navbar('B', ('Docs', '/d'))) and 'href="/d"' in to_xml(Navbar('B', ('Docs', '/d')))

## Toasts

`OatToast` shows a toast where it lands: in a page or an htmx swap. `toast_hdr` shows one after an htmx request, through `HX-Trigger`.
`setup_oat_toasts` makes FastHTML's own `add_toast(sess, msg, typ)` show Oat's toasts.

In [ ]:
#| export
def _toast(msg, title, variant, **o): return dict(msg=msg, title=title, variant={'error': 'danger'}.get(variant, variant or 'info')) | o

def OatToast(msg, title=None, variant=None, **opts) -> FT:
    "A script that shows a toast: `variant` info|success|warning|danger; opts `placement`, `duration`."
    return Script(f'(window.fhoat?.toast||(d=>window.ot?.toast(d.msg,d.title,d)))({_js(_toast(msg, title, variant, **opts))})')

def toast_hdr(msg, title=None, variant=None, events=None, **opts):
    "An `HX-Trigger` header that shows a toast after an htmx request; `events` adds your own triggers to the same header."
    return HtmxResponseHeaders(trigger=json.dumps((events or {}) | dict(toast=_toast(msg, title, variant, **opts))))

def setup_oat_toasts(app):
    "Show FastHTML's `add_toast(sess, msg, typ)` toasts with Oat."
    from fasthtml.toaster import sk
    def after(resp, req, sess):
        if sk in sess and (not resp or isinstance(resp, (tuple, FT, FtResponse))):
            req.injects.extend(OatToast(m, variant=t) for m, t, _ in sess.pop(sk, []))
    app.after.append(after)
    return app

In [ ]:
s = to_xml(OatToast('</script><script>alert(1)</script>'))
assert '</script><script>' not in s.removesuffix('</script>') and '\\u003c/script' in s
assert json.loads(toast_hdr('Saved', 'Done', 'error', events=dict(refresh=1)).v)['toast']['variant'] == 'danger'

In [ ]:
from fhoat.core import fast_app
from fasthtml.toaster import add_toast
app, rt = fast_app()
setup_oat_toasts(app)
@rt
def index(sess): add_toast(sess, 'Hi', 'success'); return P('x')
from starlette.testclient import TestClient
assert '"variant": "success"' in TestClient(app).get('/').text

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()